# Lekse 01 - Introduksjon til AI-agenter

Velkommen til den første leksen i **AI-agenter for nybegynnere**-kurset!

En **AI-agent** er et program som bruker en stor språkmodell (LLM) som sin resonneringsmotor og kan utføre *handlinger* i den virkelige verden — kalle API-er, hente data fra databaser eller kjøre kode — for å oppnå et mål på vegne av en bruker.

I denne notatboken skal du bygge din første agent: en **reiseagent** som anbefaler feriedestinasjoner. Underveis vil du lære hvordan du:

1. Knytter til Azure AI Foundry Agent Service ved hjelp av **Microsoft Agent Framework**.
2. Gir agenten et **verktøy** — en vanlig Python-funksjon den kan kalle.
3. Kjører agenten og undersøker svaret dens.
4. Strømmer agentens svar token-for-token.


## Oppsett

Før du kjører denne notatblokken, sørg for at du har:

1. **Et Azure AI Foundry-prosjekt** med en distribuert chatmodell (f.eks. `gpt-4o-mini`).
2. **Logget inn med Azure CLI** — kjør `az login` i terminalen din.
3. **Satt de nødvendige miljøvariablene:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endepunktet for Azure AI Foundry-prosjektet ditt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — navnet på den distribuerte modellen din.

Cellen nedenfor installerer Python-pakkene du trenger.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Lage din første agent

En agent trenger to ting:

- **Instruksjoner** som forteller den *hvem den er* og *hvordan den skal oppføre seg* (en systemprompt).
- **Verktøy** — Python-funksjoner dekorert med `@tool` som agenten kan kalle for å hente informasjon eller utføre handlinger.

Nedenfor definerer vi et enkelt verktøy som returnerer en liste over populære feriemål. Agenten vil bruke dette verktøyet når en bruker ber om reiseanbefalinger.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Strømming av svar

For en mer interaktiv opplevelse kan du **streame** agentens respons. I stedet for å vente på hele svaret, leverer agenten tekstbiter etter hvert som de genereres. Dette er spesielt nyttig i chattegrensesnitt hvor du ønsker å vise utdata i sanntid.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Sammendrag

I denne leksjonen lærte du hvordan du:

- **Oppretter en leverandør** som kobler til Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Definerer et verktøy** ved å bruke `@tool` dekoratøren slik at agenten kan kalle dine Python-funksjoner.
- **Kjører agenten** med en brukermelding og skriver ut svaret dens.
- **Strømmer svar** for sanntidsutdata.

I neste leksjon vil vi utforske agentiske rammeverk mer i dybden og lære hvordan vi gir agenter kraftigere verktøy og flerstegs resonnementsevner.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiserte oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket bør betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
